<a href="https://colab.research.google.com/github/Logesh8799/Hinglish-Code-Mixed-Sentiment-Classification-/blob/main/Hinglish_Code_Mixed_Analysis_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset Loading

In [ ]:
import re
import pandas as pd
import numpy as np
from collections import Counter

# 1. Load the Dataset (Handles common TSV/CSV delimiters or custom spacing)
def load_dataset(file_path):
    try:
        # Load the file, ensuring everything in the columns is treated as text strings
        df = pd.read_csv(file_path, sep='\t', names=['text', 'sentiment'], header=None, dtype=str)
    except Exception:
        df = pd.read_csv(file_path, dtype=str)

    # 1. Fill any missing rows with empty strings so the script doesn't break
    df['text'] = df['text'].fillna('')
    df['sentiment'] = df['sentiment'].fillna('unknown')

    # 2. Safely clean up the text spacing and lowercase labels
    df['sentiment'] = df['sentiment'].str.strip().str.lower()
    return df

# 2. Heuristic function to estimate language tokens per sentence
# (Used when explicit word-level language tags are not provided in the txt file)
def calculate_code_mixing_degree(text):
    # Tokenize and clean punctuation
    tokens = re.findall(r'\b\w+\b', str(text).lower())
    total_tokens = len(tokens)
    if total_tokens == 0:
        return 0.0

    # Simple dictionary heuristic for common Hindi matrix words written in Roman script (Hinglish)
    # vs standard English words.
    hinglish_stopwords = {'hai', 'ko', 'ke', 'me', 'mein', 'bhi', 'aur', 'se', 'ka', 'ki', 'tha', 'ho', 'ya', 'par'}

    hi_count = 0
    en_count = 0

    for token in tokens:
        if token in hinglish_stopwords:
            hi_count += 1
        elif len(token) > 1: # Basic fallback assumption for general english words
            en_count += 1

    # Calculate Code-Mixing Index (CMI) approximation
    # CMI = 100 * [1 - (max(lang_tokens) / total_lang_tokens)]
    max_lang = max(hi_count, en_count)
    total_lang = hi_count + en_count

    if total_lang == 0:
        return 0.0

    cmi = (1 - (max_lang / total_lang)) * 100
    return round(cmi, 2)

# 3. Main Exploration Pipeline
def explore_code_mixed_dataset(file_path):
    print(f"--- Loading data from {file_path} ---")
    df = load_dataset(file_path)

    # Class Distribution Analysis
    print("\n[1] Class Distribution Summary:")
    dist = df['sentiment'].value_counts()
    percentage = df['sentiment'].value_counts(normalize=True) * 100

    for label in dist.index:
        print(f" - {label.capitalize()}: {dist[label]} samples ({percentage[label]:.2f}%)")

    # Code-Mixing Index Analysis
    df['cmi'] = df['text'].apply(calculate_code_mixing_degree)

    print("\n[2] Degree of Code-Mixing Metrics (CMI):")
    print(f" - Average Code-Mixing Index: {df['cmi'].mean():.2f}%")
    print(f" - Maximum Code-Mixing Index: {df['cmi'].max():.2f}%")
    print(f" - Monolingual / Low-mix samples (CMI == 0): {len(df[df['cmi'] == 0])}")
    print(f" - Highly Code-Mixed samples (CMI > 30): {len(df[df['cmi'] > 30])}")

    # Display sample snippets
    print("\n[3] Highly Code-Mixed Samples Examples:")
    samples = df[df['cmi'] > 20].head(3)
    for idx, row in samples.iterrows():
        print(f" - [{row['sentiment'].upper()}] (CMI: {row['cmi']}%): \"{row['text']}\"")

# To run the code, ensure 'IIITH_Codemixed.txt' is in your path:
explore_code_mixed_dataset('IIITH_Codemixed.txt')


--- Loading data from IIITH_Codemixed.txt ---

[1] Class Distribution Summary:
 - 1: 1957 samples (50.45%)
 - 2: 1352 samples (34.85%)
 - 0: 570 samples (14.69%)

[2] Degree of Code-Mixing Metrics (CMI):
 - Average Code-Mixing Index: 0.00%
 - Maximum Code-Mixing Index: 0.00%
 - Monolingual / Low-mix samples (CMI == 0): 3879
 - Highly Code-Mixed samples (CMI > 30): 0

[3] Highly Code-Mixed Samples Examples:


PreProcessing

In [ ]:
import re
import unicodedata

# 1. Phonetic Normalization Mapping (Handles inconsistent transliteration variants)
# This maps erratic Latin-script Hindi character combinations to unified forms.
PHONETIC_REPLACEMENTS = {
    r'z': 'j',          # e.g., zindagi -> jindagi
    r'v': 'w',          # e.g., vikas -> wikas, video -> wideo
    r'oo+': 'u',        # e.g., acchaooo -> acchau, joo -> ju
    r'ee+': 'i',        # e.g., khushi -> khushi, thieek -> thik
    r'aa+': 'a',        # e.g., baaaad -> baad
    r'ai+': 'ae',       # e.g., hai -> hae
    r'kh': 'k',         # e.g., khana -> kana
    r'gh': 'g',         # e.g., ghar -> gar
    r'bh': 'b',         # e.g., bhai -> bai
    r'dh': 'd',         # e.g., dhanda -> danda
    r'th': 't',         # e.g., thik -> tik
}

# Core vocabulary anchors for word-level Language Identification (LID)
ENGLISH_LEXICON = {'the', 'and', 'to', 'of', 'in', 'is', 'you', 'that', 'it', 'he', 'was', 'for', 'on', 'are', 'as', 'with', 'his', 'they', 'i', 'at', 'be', 'this', 'have', 'from', 'good', 'bad', 'love', 'not', 'but', 'what'}
HINDI_ROMAN_LEXICON = {'hai', 'ko', 'ke', 'me', 'mein', 'bhi', 'aur', 'se', 'ka', 'ki', 'tha', 'ho', 'ya', 'par', 'na', 'ne', 'kya', 'tu', 'main', 'hum', 'hi', 'toh', 'bhai', 'yaar', 'tera', 'mera', 'kar', 'raha', 'gaya', 'aur', 'ab'}

def normalize_transliteration(text):
    """Reduces character elongation and normalizes chaotic Romanized Hindi phonetic spellings."""
    # Step A: Reduce elongated characters (e.g., "goooood" -> "good", "plzzzz" -> "plzz")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Step B: Apply structural phonetic rules sequentially
    for pattern, replacement in PHONETIC_REPLACEMENTS.items():
        text = re.sub(pattern, replacement, text)

    return text

def preprocess_code_mixed_text(text):
    """Cleans social media metadata, processes emojis, and normalizes typography."""
    if not isinstance(text, str):
        return "", [], []

    # 1. Lowercase text early
    text = text.lower().strip()

    # 2. Extract and extract Metadata
    urls = re.findall(r'https?://\S+|www\.\S+', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text) # Remove URLs

    hashtags = re.findall(r'#\w+', text)
    text = re.sub(r'#\w+', '', text) # Remove original hashtag syntax

    text = re.sub(r'@\w+', '', text) # Remove user mentions

    # 3. Extract Emojis and Demarcate them as distinct tokens
    # Using Unicode block parsing to separate emojis from glued words safely
    emoji_list = []
    cleaned_chars = []
    for char in text:
        if unicodedata.category(char) in ['So', 'Cn']: # Symbols/Emojis classification block
            emoji_list.append(char)
            cleaned_chars.append(f" {char} ") # Pad out spacing
        else:
            cleaned_chars.append(char)
    text = "".join(cleaned_chars)

    # 4. Clean up special punctuation noise while keeping alphanumeric metrics
    text = re.sub(r'[^\w\s\d\?\!\.\'\,\"\u0900-\u097F]', '', text)
    text = re.sub(r'\s+', ' ', text).strip() # Squash structural spaces

    # 5. Normalize structural spelling variants
    normalized_text = normalize_transliteration(text)

    # Tokenize for sequence alignment processing
    tokens = normalized_text.split()

    # 6. Word-Level Language Tagging Architecture
    word_tags = []
    for token in tokens:
        clean_token = re.sub(r'[^\w]', '', token)

        # Check Devanagari Block Unicode ranges explicitly
        if any('\u0900' <= c <= '\u097F' for c in clean_token):
            word_tags.append((token, 'HI_DEV')) # Native Hindi script
        elif clean_token in HINDI_ROMAN_LEXICON:
            word_tags.append((token, 'HI_ROM')) # Transliterated Hindi
        elif clean_token in ENGLISH_LEXICON:
            word_tags.append((token, 'EN'))     # Standard English
        elif clean_token in emoji_list:
            word_tags.append((token, 'EMOJI'))  # Sentimental Emojis
        elif clean_token.isdigit():
            word_tags.append((token, 'NUM'))    # Quantitative entries
        else:
            word_tags.append((token, 'MIXED_UNKN')) # Cross-boundary mixed transitions

    return normalized_text, word_tags, hashtags

# --- Execution Simulation ---
sample_tweet = "Gooood morning!! @username Yeh post bahut badhiya hai #Inspiration... Check out link https://example.com 😎👍"
cleaned_text, tags, tags_hash = preprocess_code_mixed_text(sample_tweet)

print("✨ Cleaned & Normalized Text:")
print(f"   \"{cleaned_text}\"")
print("\n🏷️ Word-level Language Identification Tags:")
print(tags)
print("\n#️⃣ Extracted Hashtags:")
print(tags_hash)


✨ Cleaned & Normalized Text:
   "gud morning!! yeh post bahut badiya hae .. check out link"

🏷️ Word-level Language Identification Tags:
[('gud', 'MIXED_UNKN'), ('morning!!', 'MIXED_UNKN'), ('yeh', 'MIXED_UNKN'), ('post', 'MIXED_UNKN'), ('bahut', 'MIXED_UNKN'), ('badiya', 'MIXED_UNKN'), ('hae', 'MIXED_UNKN'), ('..', 'MIXED_UNKN'), ('check', 'MIXED_UNKN'), ('out', 'MIXED_UNKN'), ('link', 'MIXED_UNKN')]

#️⃣ Extracted Hashtags:
['#inspiration']


Embedding & Tokenizer

In [ ]:
!pip install tokenizers gensim pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 46.3 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
from gensim.models import FastText

# 1. Prepare Text Data for Training
def prepare_corpus_file(file_path, output_txt="pure_corpus.txt"):
    """Extracts raw text sequences from your dataset to train embeddings."""
    print(" Reading and cleaning dataset rows...")
    try:
        df = pd.read_csv(file_path, sep='\t', names=['text', 'sentiment'], header=None, dtype=str)
    except Exception:
        df = pd.read_csv(file_path, dtype=str)

    df['text'] = df['text'].fillna('').str.strip()

    # Save text lines to a clean raw text file
    with open(output_txt, 'w', encoding='utf-8') as f:
        for line in df['text']:
            if line:
                f.write(line + "\n")
    return output_txt

# 2. Train a Subword BPE Tokenizer From Scratch
def train_subword_tokenizer(corpus_file, vocab_size=10000):
    """Trains a Byte-Pair Encoding (BPE) tokenizer optimized for Hinglish."""
    print(f" Training BPE Tokenizer (Vocab size: {vocab_size})...")
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
    )

    tokenizer.train([corpus_file], trainer)
    return tokenizer

# 3. Tokenize Corpus & Train FastText Embeddings
def train_codemixed_embeddings(dataset_path):
    # Step A: Clean and dump text
    corpus_file = prepare_corpus_file(dataset_path)

    # Step B: Train & compile custom subword lexicon
    tokenizer = train_subword_tokenizer(corpus_file, vocab_size=12000)

    # Step C: Tokenize the corpus into structural subword fragments
    print(" Tokenizing corpus into subword sequences...")
    tokenized_sentences = []
    with open(corpus_file, 'r', encoding='utf-8') as f:
        for line in f:
            encoded = tokenizer.encode(line.strip())
            tokenized_sentences.append(encoded.tokens)

    # Step D: Train FastText Model From Scratch
    print(" Training FastText Embedding Model (Subword N-Grams)...")
    # Vector size: 100-300 is optimal for small-to-mid scale custom text corpuses
    ft_model = FastText(
        sentences=tokenized_sentences,
        vector_size=100,
        window=5,
        min_count=2,
        workers=4,
        sg=1,            # Use Skip-gram (better for rare code-mixed variants)
        min_n=3, max_n=6 # Character n-gram range to capture structural Hinglish patterns
    )

    # Save the trained components
    ft_model.save("hinglish_fasttext.model")
    print(" Embeddings successfully trained and saved as 'hinglish_fasttext.model'!")
    return ft_model, tokenizer

# --- EXECUTE THE PIPELINE ---
# Replace 'IIITH_Codemixed.txt' with your actual Colab path if needed
ft_model, tokenizer = train_codemixed_embeddings('IIITH_Codemixed.txt')


 Reading and cleaning dataset rows...
 Training BPE Tokenizer (Vocab size: 12000)...
 Tokenizing corpus into subword sequences...
 Training FastText Embedding Model (Subword N-Grams)...
 Embeddings successfully trained and saved as 'hinglish_fasttext.model'!


 Test and Validate the Embeddings

In [ ]:
# Test semantic similarity for a common transliteration
word_to_test = "bhai"

# Note: Since the model was trained on subword tokens generated by your tokenizer,
# we verify if the token exists or evaluate structural similarities.
try:
    print(f"\nMost similar tokens to '{word_to_test}':")
    similars = ft_model.wv.most_similar(word_to_test, topn=5)
    for word, score in similars:
        print(f" - {word}: {score:.4f}")
except KeyError:
    # If the exact word string wasn't explicitly isolated, FastText uses n-grams to deduce it
    print(f"\nEvaluating subword vectors for: {word_to_test}")
    # Fallback to structural lookup
    print(ft_model.wv.get_vector(word_to_test)[:10]) # Look at first 10 vector values



Most similar tokens to 'bhai':
 - 1: 0.2412
 - 0: 0.0617
 - 2: 0.0148


LSTM

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

# 1. Configuration Constants
BATCH_SIZE = 32
EMBEDDING_DIM = 100  # Must match the vector_size from your FastText model
HIDDEN_DIM = 128
NUM_CLASSES = 3      # Positive, Negative, Neutral
MAX_LEN = 50         # Maximum sequence length for padding
EPOCHS = 5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using compute device: {DEVICE}")

# 2. Build the Embedding Weight Matrix
# Map our custom Tokenizer vocabulary indexes to FastText weights
vocab = tokenizer.get_vocab()
vocab_size = len(vocab)
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

for word, idx in vocab.items():
    # FastText handles unknown/subwords seamlessly
    embedding_matrix[idx] = ft_model.wv[word]

embedding_matrix = torch.FloatTensor(embedding_matrix)

# 3. Define the PyTorch Dataset for Code-Mixed Text
class CodeMixedDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_len):
        try:
            df = pd.read_csv(data_path, sep='\t', names=['text', 'sentiment'], header=None, dtype=str)
        except Exception:
            df = pd.read_csv(data_path, dtype=str)

        df['text'] = df['text'].fillna('').str.strip()

        # 1. Clean the label column: remove any spaces and convert to a clean string
        df['sentiment'] = df['sentiment'].fillna('').str.strip()

        # 2. Map the actual integer labels found in your dataset ('2', '1', '0')
        # This aligns them to standard target cross-entropy indexes: 0, 1, 2
        self.label_map = {
            '0': 0,  # Negative maps to index 0
            '1': 1,  # Neutral maps to index 1
            '2': 2   # Positive maps to index 2
        }

        # Keep only the rows matching our target labels
        df = df[df['sentiment'].isin(self.label_map.keys())].reset_index(drop=True)

        self.labels = [self.label_map[lbl] for lbl in df['sentiment']]
        self.sequences = []

        # Encode tokens
        for text in df['text']:
            encoded = tokenizer.encode(text).ids
            if len(encoded) < max_len:
                encoded = encoded + [0] * (max_len - len(encoded))
            else:
                encoded = encoded[:max_len]
            self.sequences.append(encoded)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.sequences[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Instantiate Dataset & DataLoader
# Replace 'IIITH_Codemixed.txt' with your file reference path if necessary
dataset = CodeMixedDataset('IIITH_Codemixed.txt', tokenizer, MAX_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# 4. Design Bi-Directional LSTM Architecture
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pretrained_embeddings):
        super(SentimentLSTM, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)

        # Corrected from nn.nn.LSTM to nn.LSTM
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)

        lstm_out, (hidden, cell) = self.lstm(embedded)

        out = torch.mean(lstm_out, dim=1)
        out = self.dropout(out)

        logits = self.fc(out)
        return logits

# Initialize Model network configurations
model = SentimentLSTM(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_CLASSES, embedding_matrix)
model = model.to(DEVICE)

# 5. Define Loss Criteria & Optimization Metrics
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 6. Execute Model Training Loop
print("\n--- Starting Sentiment LSTM Model Training ---")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    correct_preds = 0
    total_preds = 0

    for batch in dataloader:
        ids = batch['input_ids'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        # Zero execution gradients
        optimizer.zero_grad()

        # Forward pass tracking
        outputs = model(ids)
        loss = criterion(outputs, labels)

        # Backward optimization tracking
        loss.backward()
        # Gradient clipping prevents standard exploding gradients inside deeper LSTMs
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Gather metrics
        epoch_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct_preds += (predicted == labels).sum().item()
        total_preds += labels.size(0)

    avg_loss = epoch_loss / len(dataloader)
    accuracy = (correct_preds / total_preds) * 100
    print(f"Epoch {epoch+1}/{EPOCHS} -> Loss: {avg_loss:.4f} | Training Accuracy: {accuracy:.2f}%")

print("\n Training complete! The system has successfully converged.")


Using compute device: cpu

--- Starting Sentiment LSTM Model Training ---
Epoch 1/5 -> Loss: 0.8418 | Training Accuracy: 58.70%
Epoch 2/5 -> Loss: 0.2013 | Training Accuracy: 91.34%
Epoch 3/5 -> Loss: 0.0005 | Training Accuracy: 100.00%
Epoch 4/5 -> Loss: 0.0002 | Training Accuracy: 100.00%
Epoch 5/5 -> Loss: 0.0001 | Training Accuracy: 100.00%

 Training complete! The system has successfully converged.


GRU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd

# 1. Create Train, Validation, and Test Splits
# Set a random seed for reproducible, identical data partitioning
generator = torch.Generator().manual_seed(42)
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size], generator=generator)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Data split sizes -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# 2. Design Bi-Directional GRU Architecture
class SentimentGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pretrained_embeddings):
        super(SentimentGRU, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False)

        # Bidirectional GRU layer
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)

        # GRU outputs sequence hidden states
        gru_out, hidden = self.gru(embedded)

        out = torch.mean(gru_out, dim=1)
        out = self.dropout(out)
        logits = self.fc(out)
        return logits

# Initialize GRU Model
gru_model = SentimentGRU(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_CLASSES, embedding_matrix).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(gru_model.parameters(), lr=1e-3, weight_decay=1e-4)

# 3. Training & Validation Function
def evaluate_model(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            outputs = model(ids)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), (correct / total) * 100

print("\n--- Starting Sentiment GRU Model Training ---")
for epoch in range(EPOCHS):
    gru_model.train()
    epoch_loss, correct_train, total_train = 0, 0, 0

    for batch in train_loader:
        ids = batch['input_ids'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        outputs = gru_model(ids)
        loss = criterion(outputs, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(gru_model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

    train_loss = epoch_loss / len(train_loader)
    train_acc = (correct_train / total_train) * 100
    val_loss, val_acc = evaluate_model(gru_model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} -> Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% || Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

# 4. Final Evaluation on Test Set
test_loss, test_acc = evaluate_model(gru_model, test_loader)
print(f"\n Final Test Set Evaluation -> Loss: {test_loss:.4f} | Accuracy: {test_acc:.2f}%")


Data split sizes -> Train: 2715 | Val: 581 | Test: 583

--- Starting Sentiment GRU Model Training ---
Epoch 1/5 -> Train Loss: 0.6381 | Train Acc: 73.19% || Val Loss: 0.1653 | Val Acc: 82.79%
Epoch 2/5 -> Train Loss: 0.0162 | Train Acc: 99.52% || Val Loss: 0.0004 | Val Acc: 100.00%
Epoch 3/5 -> Train Loss: 0.0003 | Train Acc: 100.00% || Val Loss: 0.0002 | Val Acc: 100.00%
Epoch 4/5 -> Train Loss: 0.0002 | Train Acc: 100.00% || Val Loss: 0.0001 | Val Acc: 100.00%
Epoch 5/5 -> Train Loss: 0.0001 | Train Acc: 100.00% || Val Loss: 0.0001 | Val Acc: 100.00%

 Final Test Set Evaluation -> Loss: 0.0001 | Accuracy: 100.00%


Comparison both models

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, accuracy_score

def perform_error_analysis(model, gru_model, dataloader_split, original_dataset):
    model.eval()
    gru_model.eval()

    all_preds_lstm = []
    all_preds_gru = []
    all_labels = []

    # Extract the exact data indices handled by this random_split
    test_indices = dataloader_split.dataset.indices

    # 1. Gather all predictions across the test set loader
    with torch.no_grad():
        for batch in dataloader_split:
            ids = batch['input_ids'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            # LSTM Predictions
            outputs_lstm = model(ids)
            _, predicted_lstm = torch.max(outputs_lstm, 1)
            all_preds_lstm.extend(predicted_lstm.cpu().numpy())

            # GRU Predictions
            outputs_gru = gru_model(ids)
            _, predicted_gru = torch.max(outputs_gru, 1)
            all_preds_gru.extend(predicted_gru.cpu().numpy())

            all_labels.extend(labels.cpu().numpy())

    # 2. Reload the original raw text file to calculate CMI directly on strings
    # This avoids decoding issues with the tokenizer
    try:
        df_raw = pd.read_csv('IIITH_Codemixed.txt', sep='\t', names=['text', 'sentiment'], header=None, dtype=str)
    except Exception:
        df_raw = pd.read_csv('IIITH_Codemixed.txt', dtype=str)

    df_raw['text'] = df_raw['text'].fillna('').str.strip()

    # 3. Build analysis DataFrame for the test slice
    test_texts = [df_raw['text'].iloc[idx] for idx in test_indices]
    # Re-calculate CMI using the original string heuristic from step 1
    test_cmis = [calculate_code_mixing_degree(text) for text in test_texts]

    analysis_df = pd.DataFrame({
        'text': test_texts,
        'true_label': all_labels,
        'lstm_pred': all_preds_lstm,
        'gru_pred': all_preds_gru,
        'cmi': test_cmis
    })

    # Categorize into linguistic groups
    def categorize_cmi(cmi_val):
        if cmi_val <= 10:
            return 'Mostly Monolingual (Low Mix)'
        elif 10 < cmi_val <= 30:
            return 'Moderately Mixed'
        else:
            return 'Heavily Mixed (High Mix)'

    analysis_df['mix_category'] = analysis_df['cmi'].apply(categorize_cmi)

    # 4. Display Global Comparison
    print("\n" + "="*50)
    print(" GLOBAL CLASSIFICATION METRICS COMPARISON")
    print("="*50)
    print(f"LSTM Overall Accuracy: {accuracy_score(all_labels, all_preds_lstm)*100:.2f}%")
    print(f"GRU Overall Accuracy:  {accuracy_score(all_labels, all_preds_gru)*100:.2f}%")

    print("\n--- Detailed LSTM Classification Report ---")
    print(classification_report(all_labels, all_preds_lstm, target_names=['Negative', 'Neutral', 'Positive']))

    print("\n--- Detailed GRU Classification Report ---")
    print(classification_report(all_labels, all_preds_gru, target_names=['Negative', 'Neutral', 'Positive']))

    # 5. Display Stratified Breakdown
    print("\n" + "="*50)
    print(" STRATIFIED ERROR ANALYSIS BY LINGUISTIC MIX DEGREE")
    print("="*50)

    for cat in ['Mostly Monolingual (Low Mix)', 'Moderately Mixed', 'Heavily Mixed (High Mix)']:
        sub_df = analysis_df[analysis_df['mix_category'] == cat]
        if len(sub_df) == 0:
            continue

        y_true = sub_df['true_label']
        lstm_acc = accuracy_score(y_true, sub_df['lstm_pred']) * 100
        gru_acc = accuracy_score(y_true, sub_df['gru_pred']) * 100

        print(f"\n📊 Category: {cat} (Samples: {len(sub_df)})")
        print(f"   -> LSTM Subset Accuracy: {lstm_acc:.2f}%")
        print(f"   -> GRU Subset Accuracy:  {gru_acc:.2f}%")

    return analysis_df

# --- RUN EXECUTION ---
analysis_results = perform_error_analysis(model, gru_model, test_loader, dataset)



 GLOBAL CLASSIFICATION METRICS COMPARISON
LSTM Overall Accuracy: 100.00%
GRU Overall Accuracy:  100.00%

--- Detailed LSTM Classification Report ---
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        78
     Neutral       1.00      1.00      1.00       308
    Positive       1.00      1.00      1.00       197

    accuracy                           1.00       583
   macro avg       1.00      1.00      1.00       583
weighted avg       1.00      1.00      1.00       583


--- Detailed GRU Classification Report ---
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        78
     Neutral       1.00      1.00      1.00       308
    Positive       1.00      1.00      1.00       197

    accuracy                           1.00       583
   macro avg       1.00      1.00      1.00       583
weighted avg       1.00      1.00      1.00       583


 STRATIFIED ERROR ANALYSIS BY LINGUISTIC MI